NYC 311 Data Ingestion (Optimization Summary)

Data partitioned into 10-day chunks (90-day window) to reduce API load and improve reliability
Each chunk retrieved in 20K record batches using unique_key-based pagination to ensure ordering and avoid duplicates
Query design enforces deterministic sorting and controlled request sizes, mitigating API rate limits and timeouts

Limitation

Intermediate results stored in memory before aggregation, leading to higher RAM usage

Future Improvements

Stream-based processing (fetch → transform → store)
Parquet storage on S3 for scalability
Checkpointing for fault tolerance
Controlled parallel ingestion for improved throughput

In [16]:
#importing libraries
from sodapy import Socrata
import pandas as pd
from datetime import datetime, timedelta
import time
import json
import os

In [17]:
def update_meta_data(last_created_date,last_key):
    with open("current_state.json", "r") as f:
        state = json.load(f)

    print("LAST UPLOAD METADATA : ")
    print(state)

    state["last_created_date"] = last_created_date.isoformat()
    state["last_unique_key"] = int(last_key)
    state["last_ingest_time"] = datetime.now().isoformat()

    with open("current_state.json", "w") as f:
        json.dump(state, f, indent=4)

    print("NEW UPLOAD METADATA : ")
    print(state)

    return

In [18]:
def update_log_history(run_time,start_date,end_date,rows_loaded,files_created,duration_seconds):
    with open("load_history.json", "r") as f:
        history = json.load(f)

    print(history)

    new_record =  {
        "run_time": run_time.iso_format(),
        "load_type": "FULL",
        "start_date": start_date.iso_format(),
        "end_date": end_date.iso_format(),
        "rows_loaded": rows_loaded,
        "files_created": files_created,
        "status": "SUCCESS",
        "duration_seconds": round(duration_seconds,2)
      }

    history.append(new_record)

    with open("load_history.json", "w") as f:
        json.dump(history, f, indent=4)

    print(history)

    return

In [19]:
client = Socrata("data.cityofnewyork.us", None,timeout = 60)

#cutoff = (datetime.now() - timedelta(days=10)).strftime("%Y-%m-%dT%H:%M:%S")
#current date - 90 days is cutoff

batch_size=20000
all_dfs = []

start_date = (datetime.now() - timedelta(days=90))

start_time = time.time()

files_created = 0 
rows_loaded = 0

for i in range(9):

    chunk_start = start_date + timedelta(days=i*10)
    chunk_end   = chunk_start + timedelta(days=10)

    print(f"Loading Batch :{i} of Data -> From {(str(chunk_start))} to {str(chunk_end)}")
    last_key = 0
    
    while True:               
        results = client.get(
            "erm2-nwe9",
            where = (f"""created_date >= '{chunk_start.strftime("%Y-%m-%dT%H:%M:%S")}' 
            AND created_date < '{chunk_end.strftime("%Y-%m-%dT%H:%M:%S")}'
            AND unique_key > '{last_key}' """ ),
            order = "unique_key ASC",
            limit = batch_size
        )
    
        if not results:
            break
        
        df_batch = pd.DataFrame(results)
        all_dfs.append(df_batch)

        last_key = df_batch["unique_key"].astype(int).max()
        load_size = len(df_batch)
        
        print("\t",load_size,"Loaded , Last Key : ", last_key)
        files_created += 1
        rows_loaded += load_size

start_time2= time.time()
dfs = pd.concat(all_dfs, ignore_index=True)
end_time2= time.time()
end_time = time.time()
print("Overall Time",end_time - start_time)
print("Concat Time",end_time2 - start_time2)
#print(df_batch.shape)
print(len(all_dfs))
print(dfs.shape)

#Metadata for logs
print("Files Created : ", files_created) 
run_time = start_time 
load_type = "FULL"
end_date = start_date + timedelta(days=90)
duration_seconds = end_time - start_time

Loading Batch :0 of Data -> From 2026-03-30 07:35:28.218819 to 2026-04-09 07:35:28.218819
	 20000 Loaded , Last Key :  68523291
	 20000 Loaded , Last Key :  68544588
	 20000 Loaded , Last Key :  68565454
	 20000 Loaded , Last Key :  68586373
	 20000 Loaded , Last Key :  68609103
	 1175 Loaded , Last Key :  68745181
Loading Batch :1 of Data -> From 2026-04-09 07:35:28.218819 to 2026-04-19 07:35:28.218819
	 20000 Loaded , Last Key :  68630179
	 20000 Loaded , Last Key :  68650953
	 20000 Loaded , Last Key :  68672349
	 20000 Loaded , Last Key :  68693790
	 20000 Loaded , Last Key :  68714825
	 4169 Loaded , Last Key :  68791928
Loading Batch :2 of Data -> From 2026-04-19 07:35:28.218819 to 2026-04-29 07:35:28.218819
	 20000 Loaded , Last Key :  68740260
	 20000 Loaded , Last Key :  68761687
	 20000 Loaded , Last Key :  68782816
	 20000 Loaded , Last Key :  68803512
	 20000 Loaded , Last Key :  68833491
	 185 Loaded , Last Key :  68881642
Loading Batch :3 of Data -> From 2026-04-29 07:35:

In [20]:
#update metadata 
update_meta_data(end_date,last_key)

LAST UPLOAD METADATA : 
{'last_created_date': '2026-06-28T06:54:38.772491', 'last_unique_key': 69505203, 'last_ingest_time': '2026-06-28T07:08:34.224914'}
NEW UPLOAD METADATA : 
{'last_created_date': '2026-06-28T07:35:28.218819', 'last_unique_key': 69505203, 'last_ingest_time': '2026-06-28T07:41:47.771640'}


In [21]:
update_log_history(run_time,start_date,end_date,rows_loaded,files_created,duration_seconds)

[{'run_time': '2026-06-28T06:50:00', 'load_type': 'FULL', 'start_date': '2026-03-30', 'end_date': '2026-06-28', 'rows_loaded': 120000, 'files_created': 6, 'status': 'SUCCESS', 'duration_seconds': 45.2}, {'run_time': '2026-06-28T06:50:00', 'load_type': 'FULL', 'start_date': '2026-03-30', 'end_date': '2026-06-28', 'rows_loaded': 240000, 'files_created': 6, 'status': 'SUCCESS', 'duration_seconds': 45.2}, {'run_time': '2026-06-28T06:50:00', 'load_type': 'FULL', 'start_date': '2026-03-30', 'end_date': '2026-06-28', 'rows_loaded': 240000, 'files_created': 6, 'status': 'SUCCESS', 'duration_seconds': 45.2}]


AttributeError: 'float' object has no attribute 'iso_format'